# Detección de Muelas del Juicio — Semana 3: Entrenamiento del Modelo

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Repo:** https://github.com/JulianOliveraBalls/dentex-wisdom-teeth

---

**Prerrequisito:** haber corrido `01_dataset_preparation.ipynb` (genera `data/processed/yolo_merged/`).
Este notebook carga los pesos guardados — **no reentrena**.

| Sección | Contenido |
|---------|----------|
| 0 | Setup — detección Colab/local + restauración de pesos |
| a | Modelo preentrenado y estrategia de fine-tuning |
| b | Configuración del entrenamiento |
| c | Experimentación — 6 experimentos comparados |
| d | Evaluación del modelo elegido |
| e | Guardado del modelo |


## Sección 0 — Setup

In [9]:
import subprocess, sys, os, gc, json, random, warnings, shutil
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

try: import ultralytics, sklearn
except ImportError: pip_install('ultralytics','scikit-learn')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.metrics import classification_report
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from ultralytics import YOLO

def log(msg, level='INFO'):
    icons={'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level,"[INFO]")} {msg}')

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# ── Detección automática Colab / local ────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = os.path.exists('/content') and not os.path.exists('C:/Users')
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive; drive.mount('/drive')
    REPO_ROOT = Path('/content/dentex-wisdom-teeth')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/JulianOliveraBalls/dentex-wisdom-teeth.git',
                        str(REPO_ROOT)], check=True)
    DRIVE_DIR = Path('/drive/MyDrive/dentex_runs')
else:
    _nb_path = Path(globals().get('__vsc_ipynb_file__',
                    globals().get('__file__', str(Path.cwd() / 'dev' / 'notebook.ipynb'))))
    REPO_ROOT = _nb_path.parent.parent
    DRIVE_DIR = REPO_ROOT / 'dev'
    log(f'Modo local — REPO_ROOT: {REPO_ROOT}', 'OK')

DATA_DIR    = REPO_ROOT / 'data'
OUTPUTS_DIR = DATA_DIR / 'outputs'
SRC_DIR     = REPO_ROOT / 'src'
DEV_DIR     = REPO_ROOT / 'dev'
for d in [OUTPUTS_DIR, DEV_DIR]: d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Buscar yolo_merged en ubicaciones posibles ────────────────────────────────
_candidatos = [
    DATA_DIR / 'processed' / 'yolo_merged',
    Path('/content/data/processed/yolo_merged'),
    Path('/content/dentex-wisdom-teeth/data/processed/yolo_merged'),
]
YOLO_MERGED = None
for c in _candidatos:
    if c.exists() and len(list((c/'images'/'train').glob('*.*'))) > 100:
        YOLO_MERGED = c; break

if YOLO_MERGED is None:
    raise RuntimeError(
        'yolo_merged no encontrado.\n'
        'Correr primero 01_dataset_preparation.ipynb en esta sesión.')

log(f'REPO_ROOT: {REPO_ROOT}', 'DATA')
log(f'IN_COLAB: {IN_COLAB}   device: {device}', 'OK')
log(f'yolo_merged: {YOLO_MERGED}', 'DATA')
log(f'  train: {len(list((YOLO_MERGED/"images"/"train").glob("*.*")))} imgs', 'DATA')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).
[DATA] REPO_ROOT: /content/dentex-wisdom-teeth
[OK]   IN_COLAB: True   device: cuda
[DATA] yolo_merged: /content/dentex-wisdom-teeth/data/processed/yolo_merged
[DATA]   train: 2619 imgs


In [10]:
# ── Restaurar pesos desde Drive (Colab) o dev/ (local) ──────────────────────
RUNS_DIR = Path('runs/detect')
RESULTS  = {}

_exp_map = {
    'Exp_G1': 'Exp_G1_yolov8s_merged_30ep',
    'Exp_G2': 'Exp_G2_yolov8m_merged_30ep',
    'Exp_G4': 'Exp_G4_yolov8m_merged_60ep_cls15',
    'Exp_G5': 'Exp_G5_yolov8m_erupted_boost_60ep',
    'Exp_G6': 'Exp_G6_yolov8m_exanquad_30ep',
}
_exp_meta = {
    'Exp_G1': ('merged_2619', 30,  'YOLOv8s COCO, merged, 30ep'),
    'Exp_G2': ('merged_2619', 30,  'YOLOv8m COCO, merged, 30ep'),
    'Exp_G4': ('merged_2619', 60,  'YOLOv8m, cls=1.5, aug agresiva, 60ep'),
    'Exp_G5': ('merged_2644', 60,  'YOLOv8m, cls=1.5, +25 erupted DENTEX, 60ep'),
    'Exp_G6': ('merged_6630', 30,  'YOLOv8m, DENTEX+ExAnMTM quad_flip, 30ep'),
}

# Restaurar desde Drive → runs/ (Colab) o desde dev/ (local)
for alias, run_name in _exp_map.items():
    src_dir = DRIVE_DIR / run_name  # Drive en Colab, dev/ en local
    if not src_dir.exists(): continue
    dst_weights = RUNS_DIR / run_name / 'weights'
    dst_weights.mkdir(parents=True, exist_ok=True)
    for f in (src_dir / 'weights').glob('*.pt'):
        dst = dst_weights / f.name
        if not dst.exists(): shutil.copy2(str(f), str(dst))
    csv_src = src_dir / 'results.csv'
    csv_dst = RUNS_DIR / run_name / 'results.csv'
    if csv_src.exists() and not csv_dst.exists():
        shutil.copy2(str(csv_src), str(csv_dst))
    # Cargar métricas desde CSV
    if csv_dst.exists():
        try:
            df_r = pd.read_csv(csv_dst); df_r.columns = df_r.columns.str.strip()
            last = df_r.iloc[-1]
            dataset, epochs, notas = _exp_meta[alias]
            RESULTS[alias] = {
                'dataset': dataset, 'epochs': epochs,
                'mAP50':    round(float(last.get('metrics/mAP50(B)',   0)), 4),
                'mAP50_95': round(float(last.get('metrics/mAP50-95(B)',0)), 4),
                'P':        round(float(last.get('metrics/precision(B)',0)),4),
                'R':        round(float(last.get('metrics/recall(B)',  0)), 4),
                'notas':    notas,
            }
            log(f'{alias}: mAP50={RESULTS[alias]["mAP50"]} (epoch {len(df_r)})', 'OK')
        except Exception as e:
            log(f'{alias}: error leyendo CSV — {e}', 'WARN')

if not RESULTS:
    log('No se encontraron pesos. Copiar best.pt de cada experimento a dev/ o Drive.', 'WARN')


[OK]   Exp_G1: mAP50=0.9016 (epoch 30)
[OK]   Exp_G2: mAP50=0.9122 (epoch 30)
[OK]   Exp_G4: mAP50=0.9411 (epoch 60)
[OK]   Exp_G5: mAP50=0.9275 (epoch 60)
[OK]   Exp_G6: mAP50=0.9229 (epoch 30)


## a) Modelo preentrenado y estrategia de fine-tuning

### Modelo elegido: YOLOv8m

| Criterio | Justificación |
|----------|---------------|
| Tarea | Detección de objetos con bounding boxes |
| Tamaño | Medium (~25M params) supera Small con >2000 imágenes |
| Resolución | 640×640 — nativa de YOLOv8, necesaria para objetos pequeños |

### Backbone: COCO vs dental (nsitnov)

Se probaron dos backbones:

| Backbone | mAP50 val (10ep) | Notas |
|----------|-----------------|-------|
| nsitnov (dental) | 0.657–0.738 | Preentrenado en 8 patologías dentales |
| **COCO** (YOLOv8m.pt) | **0.743** | **Ganador** |

COCO supera al backbone dental porque nsitnov fue entrenado como segmentador —
solo ~61 de 91 capas son compatibles con la arquitectura de detección.

### Modificación de la arquitectura

Solo se cambia la cabeza de detección: `nc=80 (COCO) → nc=2 (erupted/impacted)`.
Ultralytics lo hace automáticamente al especificar `nc=2`.

### Estrategia: fine-tuning completo (sin capas congeladas)

| Decisión | Justificación |
|----------|---------------|
| Sin freeze | Con ~2600-6600 imgs y dominio similar (Rx), congelar el backbone limita la adaptación |
| lr0=0.001667 | Conservador para no destruir features preaprendidas |
| warmup=3ep | Estabiliza gradientes al inicio del fine-tuning |


## b) Configuración del entrenamiento

### Función de pérdida

YOLOv8 usa una pérdida compuesta: `loss = 7.5×box + 0.5×cls + 1.5×dfl`

| Componente | Función | Peso |
|------------|---------|------|
| box | CIoU Loss — localización precisa del bounding box | 7.5 |
| cls | Binary Cross-Entropy — clasificación erupted/impacted | 0.5 |
| dfl | Distribution Focal Loss — precisión de los bordes | 1.5 |

El desbalance erupted/impacted (~37/63%) no es severo — no requiere Focal Loss.
Se probó `cls=1.5` en Exp_G4 para penalizar más los errores en erupted,
pero empeoró el test — la BCE estándar es más estable.

### Optimizador y scheduler

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Optimizador | AdamW | Mejor convergencia que SGD en fine-tuning |
| lr0 | 0.001667 | Default ultralytics para AdamW |
| Scheduler | Cosine annealing | Decaimiento suave |
| weight_decay | 0.0005 | Regularización L2 |

### Augmentations internas de YOLOv8

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| fliplr | 0.5 | Simetría bilateral del arco dental |
| hsv_v | 0.2 | Variabilidad de brillo entre hospitales |
| degrees | 5.0 | Rotaciones por posicionamiento del paciente |
| mosaic | 0.0 | Desactivado — mezclar anatomías es inválido |



In [11]:
# Configuración base de entrenamiento
TRAIN_KWARGS = dict(
    imgsz=640, batch=8, workers=2,
    optimizer='AdamW', lr0=0.001667, lrf=0.01,
    momentum=0.937, weight_decay=0.0005, warmup_epochs=3,
    fliplr=0.5, hsv_v=0.2, degrees=5.0, translate=0.05,
    mosaic=0.0, mixup=0.0,
    plots=True, verbose=False, exist_ok=True,
)
log('TRAIN_KWARGS configurado', 'OK')
for k,v in TRAIN_KWARGS.items(): log(f'  {k}: {v}', 'DATA')


[OK]   TRAIN_KWARGS configurado
[DATA]   imgsz: 640
[DATA]   batch: 8
[DATA]   workers: 2
[DATA]   optimizer: AdamW
[DATA]   lr0: 0.001667
[DATA]   lrf: 0.01
[DATA]   momentum: 0.937
[DATA]   weight_decay: 0.0005
[DATA]   warmup_epochs: 3
[DATA]   fliplr: 0.5
[DATA]   hsv_v: 0.2
[DATA]   degrees: 5.0
[DATA]   translate: 0.05
[DATA]   mosaic: 0.0
[DATA]   mixup: 0.0
[DATA]   plots: True
[DATA]   verbose: False
[DATA]   exist_ok: True


## Resumen de experimentación

### Contexto y datasets

| Dataset | Fuente | Imágenes train | Descripción |
|---------|--------|---------------|-------------|
| `yolo_dataset` | DENTEX MICCAI 2023 | 308 | Radiografías completas, split 70/5/25 seed=42 |
| `yolo_quad` | DENTEX + cuadrantes | ~1026 | Base + 4 cuadrantes solapados por imagen (10% overlap) |
| `yolo_quad_flip` | DENTEX + cuadrantes + flip | ~1744 | Quad + flip horizontal de cada cuadrante |
| `yolo_merged` | DENTEX + ExAn-MTM | 2619 | Quad_flip DENTEX + 875 ExAn-MTM train |

**Test set:** siempre solo DENTEX (110 imgs) — nunca visto durante entrenamiento.

---

### Fase 1 — Comparación rápida (10 epochs, solo DENTEX)

**Pregunta:** ¿qué combinación de dataset y augmentation da mejor mAP50?

| Exp | Backbone | Dataset | Aug YOLOv8 | N train | mAP50 val | mAP50-95 |
|-----|----------|---------|-----------|---------|-----------|----------|
| Exp1 | COCO | base | ninguna | 308 | 0.686 | 0.450 |
| Exp2 | COCO | base | fliplr=0.5 | 308 | 0.684 | 0.443 |
| Exp3 | COCO | base | fliplr + hsv + degrees | 308 | 0.714 | 0.446 |
| Exp4 | COCO | base | scale + shear + hsv | 308 | 0.732 | 0.454 |
| Exp5 | COCO | quad | ninguna | 1026 | 0.696 | 0.464 |
| **Exp6** | **COCO** | **quad+flip** | **fliplr=0.5** | **1744** | **0.739** | **0.491** |
| Exp7 | COCO | quad+flip | fliplr + hsv + degrees | 1744 | 0.666 | 0.424 |

**Hallazgo:** Exp6 gana — los cuadrantes con flip mejoran más que la augmentation agresiva. Exp7 sugiere que aug agresiva + pocos epochs perjudica.

---

### Fase 2 — Top experimento a 30 epochs

| Exp | mAP50 (10ep) | mAP50 (30ep) | Delta |
|-----|------------|------------|-------|
| Exp6 | 0.739 | **0.785** | +0.046 |

---

### Fase 3 — Backbones alternativos (50 epochs, solo DENTEX)

**Pregunta:** ¿un backbone preentrenado en patologías dentales supera a COCO?

| Exp | Backbone | Dataset | Epochs | mAP50 val | mAP50-95 |
|-----|----------|---------|--------|-----------|----------|
| Exp6 (referencia) | COCO | quad+flip 1744 | 50 | 0.738 | 0.479 |
| Exp9 — jerárquico | COCO → dental | base 308 | 30/etapa | 0.710 | — |
| **Exp10** | **nsitnov dental** | **base 308** | **50** | **0.769** | **0.530** |
| Exp11 | nsitnov dental | quad+flip 1744 | 45 (ES) | 0.758 | 0.487 |
| Exp12 | nsitnov frozen (10 capas) | quad+flip 1744 | 36 (ES) | 0.748 | 0.519 |

**Hallazgos:**
- **Exp10 gana** — nsitnov con dataset base supera a COCO con dataset expandido
- Más datos no ayuda con backbone dental (Exp11 < Exp10): los cuadrantes son crops de las mismas 308 imágenes, el modelo memoriza sin generalizar
- Congelar el backbone no ayuda (Exp12 < Exp11): las features dentales de nsitnov necesitan adaptarse al dominio erupted/impacted

---

### Fase G — Dataset merged DENTEX + ExAn-MTM

**Pregunta:** ¿agregar ExAn-MTM mejora los resultados? ¿Cómo tratar erupted?

| Exp | Modelo | Dataset | Epochs | Cambio respecto anterior | mAP50 val | mAP50 test | erupted test | impacted test | R test |
|-----|--------|---------|--------|--------------------------|-----------|------------|-------------|--------------|--------|
| Exp_G1 | YOLOv8s | merged 2619 | 30 | cambio a YOLOv8s + ExAn-MTM | 0.930 | — | — | — | 0.828 |
| **Exp_G2** | **YOLOv8m** | **merged 2619** | **30** | **YOLOv8m en lugar de s** | **0.940** | **0.675** | **0.444** | **0.905** | **0.917** |
| Exp_G4 | YOLOv8m | merged 2619 | 60 | cls=1.5, aug más agresiva, +30ep | 0.969 | 0.672 | 0.441 | 0.903 | 0.759 |
| Exp_G5 | YOLOv8m | merged 2644 | 60 | +25 imágenes erupted DENTEX en train | 0.954 | 0.614 | 0.145 | 0.572 | 0.864 |
| Exp_G6 | YOLOv8m | merged 6630 | 30 | ExAn-MTM también expandido con cuadrantes | 0.956 | 0.668 | 0.391 | **0.945** | 0.771 |

**Hallazgos:**
- **G2 es el mejor balance** — G6 mejora impacted (+0.04) pero empeora erupted (-0.05)
- cls=1.5 mejora val pero no test → overfitting al val set de ExAn-MTM (G4)
- Mover erupted de test a train colapsa impacted → resultado inválido (G5)
- La caída val→test (0.940→0.675) es domain gap, no overfitting clásico

---

### Modelo final: Exp_G2

**Criterio:** mejor balance erupted/impacted en test set (solo DENTEX).

| Clase | mAP50 test | P | R | F1 |
|-------|-----------|---|---|----|
| erupted | 0.444 | 0.398 | 0.452 | 0.424 |
| **impacted** | **0.905** | **0.794** | **0.939** | **0.860** |
| all | 0.675 | 0.596 | 0.917 | 0.723 |

**Limitación documentada:** erupted requiere contexto global del arco dental (saber que es el diente 8 de la numeración FDI). YOLO trabaja con ventanas locales y no puede contar dientes desde el centro del arco — limitación estructural de la arquitectura, no del dataset.

## Tabla comparativa de experimentos — decisiones y cambios

| Exp | Backbone | Dataset | N train | Epochs | Aug extra | cls weight | freeze | mAP50 val | mAP50 test | erupted test | impacted test |
|-----|----------|---------|---------|--------|-----------|------------|--------|-----------|------------|-------------|--------------|
| Exp1 | COCO | base | 308 | 10 | — | 0.5 | — | 0.686 | — | — | — |
| Exp2 | COCO | base | 308 | 10 | fliplr=0.5 | 0.5 | — | 0.684 | — | — | — |
| Exp3 | COCO | base | 308 | 10 | fliplr+hsv+degrees | 0.5 | — | 0.714 | — | — | — |
| Exp4 | COCO | base | 308 | 10 | scale+shear+hsv | 0.5 | — | 0.732 | — | — | — |
| Exp5 | COCO | quad | 1026 | 10 | — | 0.5 | — | 0.696 | — | — | — |
| Exp6 | COCO | quad+flip | 1744 | 10→30→50 | fliplr=0.5 | 0.5 | — | 0.739→0.785→0.738 | — | — | — |
| Exp7 | COCO | quad+flip | 1744 | 10 | fliplr+hsv+degrees | 0.5 | — | 0.666 | — | — | — |
| Exp9 | COCO→dental | base | 308 | 30/etapa | fliplr=0.5 | 0.5 | — | 0.710 | — | — | — |
| Exp10 | nsitnov | base | 308 | 50 | fliplr=0.5 | 0.5 | — | **0.769** | — | — | — |
| Exp11 | nsitnov | quad+flip | 1744 | 45 (ES) | fliplr=0.5 | 0.5 | — | 0.758 | — | — | — |
| Exp12 | nsitnov | quad+flip | 1744 | 36 (ES) | fliplr=0.5 | 0.5 | 10 capas | 0.748 | — | — | — |
| Exp_G1 | COCO | merged | 2619 | 30 | fliplr=0.5 | 0.5 | — | 0.930 | — | — | — |
| **Exp_G2** | **COCO** | **merged** | **2619** | **30** | **fliplr=0.5** | **0.5** | **—** | **0.940** | **0.675** | **0.444** | **0.905** |
| Exp_G4 | COCO | merged | 2619 | 60 | fliplr+hsv+degrees+scale | **1.5** | — | 0.969 | 0.672 | 0.441 | 0.903 |
| Exp_G5 | COCO | merged+25eru | **2644** | 60 | fliplr+hsv+degrees+scale | 1.5 | — | 0.954 | 0.614 | 0.145 | 0.572 |
| Exp_G6 | COCO | merged+ExAnQuad | **6630** | 30 | fliplr=0.5 | 0.5 | — | 0.956 | 0.668 | 0.391 | **0.945** |

## d) Evaluación del modelo elegido: Exp_G2

**Criterio de elección:** mejor mAP50 global en test. G6 mejora impacted (+0.04)
pero empeora erupted (-0.05) — G2 tiene el mejor balance.


### Métricas finales sobre test set

| Clase | mAP50 | P | R | F1 |
|-------|-------|---|---|----|
| erupted | 0.444 | 0.398 | 0.452 | 0.424 |
| **impacted** | **0.905** | **0.794** | **0.939** | **0.860** |
| all | 0.675 | 0.596 | 0.917 | 0.723 |

### Clasificación por imagen

| Clase | P | R | F1 | Support |
|-------|---|---|----|---------|
| erupted | 0.846 | 0.667 | 0.746 | 33 |
| impacted | 0.810 | 0.922 | 0.862 | 51 |
| **accuracy** | | | **0.821** | **84** |

La diferencia entre mAP50=0.444 y F1=0.746 en erupted indica que el modelo
**clasifica correctamente** pero a veces el bounding box no tiene suficiente IoU.



Val set pequeño eso las oscilaciones

## e) Guardado del modelo

Los pesos del modelo final se guardan en `dev/modelo.pth` y `dev/Exp_G2_best.pt`.

**Tamaño:** ~52 MB — dentro del límite de 100 MB de GitHub (no requiere Git LFS).

Si se usara YOLOv8l (>100 MB) habría que activar LFS:
```bash
git lfs install && git lfs track '*.pt' '*.pth'
```


In [12]:
# ── Exp_G7: Todo junto — máxima data + aug offline + test ampliado ────────────

# 1. Generar augmentations offline sobre yolo_merged (solo transforms safe)
# Safe = no geométricas → no invalidan boxes
import sys
sys.path.insert(0, str(SRC_DIR))
from augmentations_config import AUGMENTATIONS
import torchvision.transforms as T

MEAN_T = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD_T  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

YOLO_G7 = DATA_DIR / 'processed' / 'yolo_g7'

def generar_aug_offline(src, dst, aug_names=('C_color', 'F_mixaug')):
    """
    Copia el dataset src a dst y agrega versiones aumentadas de train
    usando solo augmentations que NO mueven píxeles (safe para boxes).
    C_color: ColorJitter + RandomAffine(5°) — affine leve, aceptable
    F_mixaug: ColorJitter + GaussianBlur + RandomErasing
    """
    # Copiar val y test sin cambios
    for split in ['val', 'test']:
        for sub in ['images', 'labels']:
            (dst/sub/split).mkdir(parents=True, exist_ok=True)
            for f in (src/sub/split).glob('*.*'):
                lnk = dst/sub/split/f.name
                if not lnk.exists(): os.symlink(f.resolve(), lnk)

    # Train: copiar originales + generar versiones aumentadas
    (dst/'images'/'train').mkdir(parents=True, exist_ok=True)
    (dst/'labels'/'train').mkdir(parents=True, exist_ok=True)
    n_orig = n_aug = 0

    for img_path in sorted(list((src/'images'/'train').glob('*.png')) +
                            list((src/'images'/'train').glob('*.jpg'))):
        lbl_path = src/'labels'/'train'/(img_path.stem+'.txt')
        if not lbl_path.exists(): continue

        # Original
        di = dst/'images'/'train'/img_path.name
        dl = dst/'labels'/'train'/lbl_path.name
        if not di.exists():
            real = Path(os.path.realpath(str(img_path)))
            if real.exists() and real != di: os.symlink(real, di)
            else: shutil.copy2(str(img_path), str(di))
        if not dl.exists(): shutil.copy2(str(lbl_path), str(dl))
        n_orig += 1

        # Versiones aumentadas
        pil = Image.open(img_path).convert('RGB')
        for aug_name in aug_names:
            transform = AUGMENTATIONS[aug_name]
            aug_t = transform(pil)
            # Desnormalizar → PIL para guardar
            aug_img = (aug_t * STD_T + MEAN_T).clamp(0, 1)
            aug_pil = T.ToPILImage()(aug_img)
            stem = f'{img_path.stem}_aug_{aug_name}'
            aug_pil.save(str(dst/'images'/'train'/(stem+'.png')))
            # Mismo label — boxes válidos porque no hay transforms geométricas
            shutil.copy2(str(lbl_path), str(dst/'labels'/'train'/(stem+'.txt')))
            n_aug += 1

    # dataset.yaml con test ampliado
    (dst/'dataset.yaml').write_text(
        f'path: {dst}\ntrain: images/train\nval: images/val\n'
        f'test: images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')
    return n_orig, n_aug

def split_existe(d):
    return Path(d).exists() and all(
        len(list((Path(d)/'images'/s).glob('*.*'))) > 0
        for s in ['train','val','test'])

if split_existe(YOLO_G7) and len(list((YOLO_G7/'images'/'train').glob('*aug*'))) > 0:
    n_train = len(list((YOLO_G7/'images'/'train').glob('*.*')))
    log(f'yolo_g7 ya existe: {n_train} imgs train', 'OK')
else:
    if YOLO_G7.exists(): shutil.rmtree(str(YOLO_G7))
    log('Generando yolo_g7 con aug offline...', 'WARN')
    n_orig, n_aug = generar_aug_offline(YOLO_MERGED, YOLO_G7)
    n_train = len(list((YOLO_G7/'images'/'train').glob('*.*')))
    log(f'yolo_g7: {n_orig} orig + {n_aug} aug = {n_train} imgs train', 'OK')

[WARN] Generando yolo_g7 con aug offline...
[OK]   yolo_g7: 2619 orig + 5238 aug = 7857 imgs train


In [13]:
# 2. Agregar ExAn-MTM val como test adicional
EXAN_VAL_IMG = DATA_DIR / 'raw' / 'exan_mtm' / 'ExAn-MTM dataset' / 'valid' / 'images'
EXAN_VAL_LBL = DATA_DIR / 'raw' / 'exan_mtm' / 'ExAn-MTM dataset' / 'valid' / 'labels'

test_dir_g7 = YOLO_G7 / 'images' / 'test'
test_lbl_g7 = YOLO_G7 / 'labels' / 'test'

n_exan_test = 0
if EXAN_VAL_IMG.exists():
    for img_p in sorted(list(EXAN_VAL_IMG.glob('*.jpg')) + list(EXAN_VAL_IMG.glob('*.png'))):
        lbl_p = EXAN_VAL_LBL / (img_p.stem + '.txt')
        if not lbl_p.exists(): continue
        di = test_dir_g7 / f'exan_{img_p.name}'
        dl = test_lbl_g7 / f'exan_{img_p.stem}.txt'
        if not di.exists(): os.symlink(img_p.resolve(), di)
        if not dl.exists(): shutil.copy2(str(lbl_p), str(dl))
        n_exan_test += 1

n_test_total = len(list(test_dir_g7.glob('*.*')))
log(f'Test ampliado: {n_test_total} imgs (DENTEX + {n_exan_test} ExAn-MTM)', 'OK')

[OK]   Test ampliado: 208 imgs (DENTEX + 98 ExAn-MTM)


In [14]:
# Verificar que yolo_g7 quedó bien construido
print(f'Train: {len(list((YOLO_G7/"images"/"train").glob("*.*")))} imgs')
print(f'Val:   {len(list((YOLO_G7/"images"/"val").glob("*.*")))} imgs')
print(f'Test:  {len(list((YOLO_G7/"images"/"test").glob("*.*")))} imgs')

# Verificar que hay imágenes aug_ en train
n_aug = len(list((YOLO_G7/"images"/"train").glob("*aug*")))
n_orig = len(list((YOLO_G7/"images"/"train").glob("*.*"))) - n_aug
print(f'  orig: {n_orig}  aug: {n_aug}')

Train: 7857 imgs
Val:   120 imgs
Test:  208 imgs
  orig: 2619  aug: 5238


In [15]:
# Verificar distribución por clase en cada split
for split in ['train', 'val', 'test']:
    lbl_dir = YOLO_G7 / 'labels' / split
    c0 = c1 = 0
    for lbl in lbl_dir.glob('*.txt'):
        for line in lbl.read_text().strip().split('\n'):
            if line:
                if int(line.split()[0]) == 0: c0 += 1
                else: c1 += 1
    total = max(c0+c1, 1)
    log(f'{split:5s}: {len(list(lbl_dir.glob("*.txt")))} imgs | erupted={c0} ({c0/total*100:.0f}%) impacted={c1} ({c1/total*100:.0f}%)', 'DATA')

[DATA] train: 7857 imgs | erupted=4269 (37%) impacted=7137 (63%)
[DATA] val  : 120 imgs | erupted=80 (35%) impacted=147 (65%)
[DATA] test : 208 imgs | erupted=147 (37%) impacted=248 (63%)


In [16]:
TRAIN_KWARGS = dict(
    imgsz=640, batch=8, workers=2,
    optimizer='AdamW', lr0=0.001667, lrf=0.01,
    momentum=0.937, weight_decay=0.0005, warmup_epochs=3,
    fliplr=0.5, hsv_v=0.2, degrees=5.0, translate=0.05,
    mosaic=0.0, mixup=0.0,
    plots=True, verbose=False, exist_ok=True,
)
log('TRAIN_KWARGS configurado', 'OK')

[OK]   TRAIN_KWARGS configurado


In [17]:
import shutil, threading, time
from pathlib import Path
from ultralytics import YOLO

# Restaurar last.pt desde Drive
run_dst = Path('runs/detect/Exp_G7_todo_junto/weights')
run_dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(
    str(DRIVE_DIR / 'Exp_G7_todo_junto' / 'weights' / 'last.pt'),
    str(run_dst / 'last.pt'))
log(f'last.pt restaurado: {(run_dst/"last.pt").stat().st_size/1e6:.1f} MB', 'OK')

csv_src = DRIVE_DIR / 'Exp_G7_todo_junto' / 'results.csv'
csv_dst = Path('runs/detect/Exp_G7_todo_junto/results.csv')
csv_dst.parent.mkdir(parents=True, exist_ok=True)
if csv_src.exists():
    shutil.copy2(str(csv_src), str(csv_dst))
    log('results.csv restaurado', 'OK')

# Autoguardado — esta vez incluye best.pt
def autoguardar_drive(run_name, drive_dir, intervalo_min=15):
    run_path = Path('runs/detect') / run_name
    dst = Path(drive_dir) / run_name
    (dst / 'weights').mkdir(parents=True, exist_ok=True)
    while True:
        time.sleep(intervalo_min * 60)
        try:
            for fname in ['last.pt', 'best.pt']:
                src = run_path / 'weights' / fname
                if src.exists():
                    shutil.copy2(str(src), str(dst / 'weights' / fname))
            csv = run_path / 'results.csv'
            if csv.exists():
                shutil.copy2(str(csv), str(dst / 'results.csv'))
            log(f'Checkpoint guardado en Drive', 'OK')
        except Exception as e:
            log(f'Error autoguardado: {e}', 'WARN')

t = threading.Thread(
    target=autoguardar_drive,
    args=('Exp_G7_todo_junto', str(DRIVE_DIR)),
    daemon=True)
t.start()
log('Autoguardado cada 15 min activado (last.pt + best.pt)', 'OK')

# Retomar
model_G7 = YOLO(str(run_dst / 'last.pt'))
results_G7 = model_G7.train(resume=True)

[OK]   last.pt restaurado: 207.4 MB
[OK]   results.csv restaurado
[OK]   Autoguardado cada 15 min activado (last.pt + best.pt)
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dentex-wisdom-teeth/data/processed/yolo_g7/dataset.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001667, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

In [18]:
# Guardar best.pt al terminar
dst_drive = DRIVE_DIR / 'Exp_G7_todo_junto'
(dst_drive / 'weights').mkdir(parents=True, exist_ok=True)

for fname in ['best.pt', 'last.pt']:
    src = Path('runs/detect/Exp_G7_todo_junto/weights') / fname
    if src.exists():
        shutil.copy2(str(src), str(dst_drive / 'weights' / fname))
        log(f'{fname} guardado en Drive: {src.stat().st_size/1e6:.1f} MB', 'OK')

csv_src = Path('runs/detect/Exp_G7_todo_junto/results.csv')
if csv_src.exists():
    shutil.copy2(str(csv_src), str(dst_drive / 'results.csv'))
    log('results.csv guardado en Drive', 'OK')

log(f'Exp_G7 completo — Drive: {dst_drive}', 'OK')

[OK]   last.pt guardado en Drive: 52.0 MB
[OK]   results.csv guardado en Drive
[OK]   Exp_G7 completo — Drive: /drive/MyDrive/dentex_runs/Exp_G7_todo_junto


In [24]:
from ultralytics import YOLO
from pathlib import Path

def log(msg, level='INFO'):
    icons = {'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level,"[INFO]")} {msg}')

DRIVE_DIR = Path('/content/gdrive/MyDrive/dentex_runs')
YOLO_G7   = Path('/content/dentex-wisdom-teeth/data/processed/yolo_g7')

weights = DRIVE_DIR / 'Exp_G7_todo_junto' / 'weights' / 'last.pt'
log(f'Usando: {weights.name}  ({weights.stat().st_size/1e6:.1f} MB)', 'OK')

model_G7 = YOLO(str(weights))
metrics_G7 = model_G7.val(
    data=str(YOLO_G7 / 'dataset.yaml'),
    split='test', imgsz=640, batch=8, verbose=True)

log('Exp_G7 last.pt — Test (208 imgs):', 'DATA')
log(f'  mAP50 all:      {metrics_G7.box.map50:.3f}', 'DATA')
log(f'  mAP50 erupted:  {metrics_G7.box.ap50[0]:.3f}', 'DATA')
log(f'  mAP50 impacted: {metrics_G7.box.ap50[1]:.3f}', 'DATA')
log(f'  P: {metrics_G7.box.mp:.3f}  R: {metrics_G7.box.mr:.3f}', 'DATA')

[OK]   Usando: last.pt  (52.0 MB)
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 724.5±551.9 MB/s, size: 1423.2 KB)
val: Scanning /content/dentex-wisdom-teeth/data/processed/yolo_g7/labels/test... 208 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 208/208 65.6it/s 3.2s<0.1s
val: New cache created: /content/dentex-wisdom-teeth/data/processed/yolo_g7/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 2.5it/s 10.5s0.2s
                   all        208        395      0.793      0.777      0.756      0.428
               erupted         99        147      0.641      0.646      0.561      0.301
              impacted        127        248      0.945      0.907      0.951      0.554
Speed: 1.9ms preprocess, 14.0ms inference, 0.0ms loss, 2.0m

[WARN] Error autoguardado: [Errno 107] Transport endpoint is not connected: '/drive/MyDrive/dentex_runs/Exp_G7_todo_junto/weights/last.pt'


In [10]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── imagen desde Drive ────────────────────────────────────────────
img_path = Path('/content/gdrive/MyDrive/dentex_runs/Exp_G7_todo_junto/myrad.jpeg')
img = cv2.imread(str(img_path))
assert img is not None, f'[ERR] No se encontró: {img_path}'
print(f'[OK] {img_path.name}  {img.shape[1]}x{img.shape[0]}px')

# ── inferencia ────────────────────────────────────────────────────
results = model_G7.predict(source=img, conf=0.25, imgsz=640, verbose=False)[0]

# ── dibujar solo impacted ─────────────────────────────────────────
img_draw   = img.copy()
detections = []

for box in results.boxes:
    if int(box.cls.item()) != 1:
        continue
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf_val = box.conf.item()
    detections.append((x1, y1, x2, y2, conf_val))

    cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 80, 220), 2)
    label = f'impacted {conf_val:.2f}'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(img_draw, (x1, y1-th-6), (x1+tw+4, y1), (0, 80, 220), -1)
    cv2.putText(img_draw, label, (x1+2, y1-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

# ── mostrar ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(cv2.cvtColor(img_draw, cv2.COLOR_BGR2RGB))
ax.set_title(f'{img_path.name}  ·  {len(detections)} muela(s) impactada(s)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

for i, (x1,y1,x2,y2,c) in enumerate(detections, 1):
    print(f'  [{i}] bbox=({x1},{y1},{x2},{y2})  conf={c:.3f}')

if not detections:
    print('[INFO] Sin detecciones — probá bajar conf a 0.15')

AssertionError: [ERR] No se encontró: /content/gdrive/MyDrive/dentex_runs/Exp_G7_todo_junto/myrad.jpeg